In [ ]:
# General notebook settings
import logging
import warnings

import pypsa

warnings.filterwarnings("error", category=DeprecationWarning)
# pandas<3.0.3 sets the `locs` attribute deprecated in matplotlib>=3.11
warnings.filterwarnings("ignore", message="The locs attribute was deprecated")
logging.getLogger("gurobipy").propagate = False
pypsa.options.params.optimize.log_to_console = False

# Piecewise capital costs with modular and semi-continuous expansion

A piecewise capital cost curve can be combined with each non-continuous `build_mode`: `"modular"` and `"semi_continuous"`.
In both cases, the curve gives the total capital cost of the asset as a function of its total capacity `p_nom`, not a cost per unit of capacity, and is zero at zero capacity.
Any cost that does not scale with capacity, such as a grid connection fee or a fixed cost per module, goes in `build_cost`.
The capital expenditure of an asset is then the total capital cost read from the curve at `p_nom`, plus `build_cost` times `n_mod`.

For the piecewise capital cost API, see the [piecewise costs and constraints example](piecewise-constraints.ipynb).
For modular expansion on its own, see the [modular expansion example](modular-expansion.ipynb).

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})

We use a single-node network with a daily load profile over 24 hourly snapshots.
An existing peaking plant covers any demand that new capacity does not, at a high marginal cost.

In [ ]:
def base_network() -> pypsa.Network:
    n = pypsa.Network(snapshots=range(24))
    n.add("Carrier", ["electricity", "gas", "solar", "oil"])
    n.add("Bus", "bus", carrier="electricity")
    load = 100 + 40 * np.sin(np.linspace(0, 2 * np.pi, 24))
    n.add("Load", "load", bus="bus", p_set=load)
    n.add("Generator", "peaker", bus="bus", carrier="oil", p_nom=200, marginal_cost=200)
    return n


def cost_summary(n: pypsa.Network) -> pd.Series:
    """splits the capital expenditure from `n.statistics.capex()` into the capacity term and the `build_cost` term.

    It then asserts that the two parts add up to the total capital expenditure, and that capital plus operational expenditure equals the objective.
    The capacity term includes the piecewise curve, which PyPSA stores per unit of capacity in `capital_cost_piecewise_opt`.
    """
    summary = pd.Series(
        {
            "capital_cost": n.statistics.capex(cost_attribute="capital_cost").sum(),
            "build_cost": n.statistics.capex(cost_attribute="build_cost").sum(),
            "capex": n.statistics.capex().sum(),
            "opex": n.statistics.opex().sum(),
            "objective": n.objective,
        }
    )
    assert np.isclose(summary["capex"], summary["capital_cost"] + summary["build_cost"])
    assert np.isclose(summary["objective"], summary["capex"] + summary["opex"])
    return summary

In [ ]:
CARRIER_COLORS = {"gas": "#2a78d6", "oil": "#eb6834", "solar": "#eda100"}
COST_COLORS = {"capital_cost": "#1baf7a", "build_cost": "#e87ba4", "opex": "#6250d6"}


def plot_build_mode(ax: plt.Axes, n: pypsa.Network, name: str, title: str) -> None:
    """Plot the impact of build mode on the capital expenditure of a generator."""
    gen = n.generators.loc[name]
    piecewise = n.c.generators.piecewise.capital_cost
    if name in piecewise:
        per_mw = piecewise[name].dropna().set_index("p_nom")["capital_cost"]
    else:
        per_mw = pd.Series({0: 0, gen.p_nom_max: gen.capital_cost})
    curve = (per_mw * per_mw.index.to_series().diff().fillna(0)).cumsum() / 1e3
    p_nom_max = min(gen.p_nom_max, curve.index[-1])
    if gen.build_mode == "modular":
        capacity = np.arange(0, p_nom_max + 1, gen.p_nom_mod)
        n_mod = capacity / gen.p_nom_mod
        style = "o"
    else:
        capacity = np.r_[0, np.linspace(gen.p_nom_min, p_nom_max, 40)]
        n_mod = capacity > 0
        style = "-"
    capex = np.interp(capacity, curve.index, curve) + gen.build_cost / 1e3 * n_mod
    capex_opt = (
        n.statistics.capex(components="Generator", groupby="name").get(name, 0) / 1e3
    )

    ax.plot(curve.index, curve, "--", color="0.6", label="capital cost alone")
    ax.plot(0, 0, "o", color="#2a78d6", markersize=4)
    ax.plot(
        capacity[1:],
        capex[1:],
        style,
        color="#2a78d6",
        markersize=4,
        linewidth=3,
        label="allowed",
    )
    ax.plot(
        gen.p_nom_opt,
        capex_opt,
        "o",
        color="black",
        markersize=14,
        fillstyle="none",
        markeredgewidth=2,
        label="optimum",
    )
    ax.set(
        title=title,
        xlabel="Capacity [MW]",
        ylabel="Capital expenditure [k€]",
        ylim=(0, None),
    )
    ax.legend(loc="upper left", fontsize="small")


def plot_dispatch(ax: plt.Axes, n: pypsa.Network, title: str) -> None:
    """plot the stack of generator dispatch vs the load over a day."""
    dispatch = n.generators_t.p.iloc[:, ::-1].loc[:, lambda df: df.sum() > 1e-6]
    colors = n.generators.carrier[dispatch.columns].map(CARRIER_COLORS)
    ax.stackplot(dispatch.index, dispatch.T, labels=dispatch.columns, colors=colors)
    ax.plot(
        n.snapshots, n.loads_t.p_set["load"], color="black", linewidth=2, label="load"
    )
    ax.set(title=title, xlabel="Hour", ylabel="Power [MW]", xlim=(0, 23), ylim=(0, 160))
    ax.legend(loc="lower left", fontsize="small")


def plot_costs(ax: plt.Axes, summaries: dict[str, pd.Series]) -> None:
    """plot the breakdown of total costs into capital, build, and operational expenditure, as returned by `cost_summary`."""
    costs = pd.DataFrame(summaries).T[list(COST_COLORS)] / 1e3
    costs.plot.bar(
        ax=ax,
        stacked=True,
        color=COST_COLORS,
        edgecolor="white",
        linewidth=2,
        rot=0,
        width=0.6,
    )
    ax.set(ylabel="Cost [k€]", title="Objective")
    ax.legend(fontsize="small", loc="upper left", bbox_to_anchor=(1, 1))

## Modular expansion with a fleet discount

Let's say that a wind farm can be built in batches of 20 MW (`build_mode="modular"`, `p_nom_mod=20`).
Each batch (in PyPSA - a "module") also has a fixed cost of 2,000 € in `build_cost` - the cost of land purchase.

With a linear `capital_cost`, every module costs the same.
With a concave piecewise curve, each additional module is cheaper than the one before.
This represents a fleet discount, for instance from a bulk purchase or from learning across identical units.

We place the breakpoints at multiples of the module size.
The capital cost is a marginal value, so the value at breakpoint $20k$ MW is the cost per MW of the $k$-th module.
The first module costs 1,000 €/MW and the seventh costs 400 €/MW.

In [ ]:
fleet_discount = {
    0: 0,
    20: 1000,
    40: 800,
    60: 650,
    80: 550,
    100: 500,
    120: 450,
    140: 400,
}


def add_wind_generator(n: pypsa.Network, capital_cost: float | dict) -> None:
    n.add(
        "Generator",
        "onshore-wind",
        bus="bus",
        carrier="wind",
        marginal_cost=50,
        p_nom_extendable=True,
        build_mode="modular",
        p_nom_mod=20,
        p_nom_max=140,
        build_cost=2000,
        capital_cost=capital_cost,
    )


n_linear = base_network()
add_wind_generator(n_linear, capital_cost=1000)
n_linear.optimize()

n_fleet = base_network()
add_wind_generator(n_fleet, capital_cost=fleet_discount)
n_fleet.optimize();

The left panel shows the input capital cost per MW of each module for both networks.
The other two panels show the capital expenditure for each number of modules: the area under the left curve plus 2,000 € of `build_cost` per module.
Only multiples of 20 MW are allowed, so these panels show points.
The black rings are the capacity that each network builds and its capital expenditure, and they sit on the allowed points.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

capacity = np.arange(0, 160, 20)
axes[0].stairs(
    np.full(7, 1000),
    capacity,
    baseline=None,
    color="0.5",
    linestyle="--",
    label="linear",
)
axes[0].stairs(
    list(fleet_discount.values())[1:],
    capacity,
    baseline=None,
    color="#2a78d6",
    label="fleet discount",
)
axes[0].set(
    xlabel="Capacity [MW]",
    ylabel="Capital cost [€/MW]",
    ylim=(0, 1100),
    title="Cost per MW of each module",
)
axes[0].legend(loc="lower left")

plot_build_mode(axes[1], n_linear, "onshore-wind", "Linear")
plot_build_mode(axes[2], n_fleet, "onshore-wind", "Fleet discount")
axes[2].sharey(axes[1])

fig.tight_layout()

The fleet discount makes the later modules cheap enough to displace more of the peaking plant.
So, the model builds seven modules (140MW) instead of six (120MW).

In both networks, `build_cost` contributes 2,000 € for each module built.
With the fleet discount, the capacity term is the area under the curve up to `p_nom`.

The figure shows the hourly dispatch of both networks and the contributors to total cost, the minimisation of which was the objective of our optimisation.
The fleet discount lowers the capital expenditure even though the network builds a seventh module.
The seventh module replaces the peaking plant at the daily peak, so the operational expenditure also falls.

In [ ]:
summary_linear = cost_summary(n_linear)
summary_fleet = cost_summary(n_fleet)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), width_ratios=[2, 2, 1.2])
plot_dispatch(axes[0], n_linear, "Linear: 6 modules")
plot_dispatch(axes[1], n_fleet, "Fleet discount: 7 modules")
plot_costs(axes[2], {"linear": summary_linear, "fleet\ndiscount": summary_fleet})
fig.tight_layout()

## Semi-continuous expansion with a build cost

Now let's say we have a solar park with a binary build decision (`build_mode="semi_continuous"`).
If it is built, its capacity must lie between `p_nom_min` (40 MW) and `p_nom_max` (150 MW).
Its capital cost per MW rises with capacity, because larger parks need more expensive land.
A grid connection fee goes in `build_cost` and is only paid if the park is built.

Now, let's compare two cases where the build cost varies sufficiently to cross a threshold between the solar park being built or not.

In [ ]:
solar_profile = np.clip(np.sin(np.linspace(-np.pi / 2, 3 * np.pi / 2, 24)), 0, None)
land_cost = {0: 0, 50: 300, 100: 500, 150: 800}


def solar_network(
    build_cost: float, capital_cost: dict = land_cost, p_nom_max: float = 150
) -> pypsa.Network:
    n = base_network()
    n.add(
        "Generator",
        "solar",
        bus="bus",
        carrier="solar",
        p_max_pu=solar_profile,
        p_nom_extendable=True,
        build_mode="semi_continuous",
        p_nom_min=40,
        p_nom_max=p_nom_max,
        build_cost=build_cost,
        capital_cost=capital_cost,
    )
    n.optimize()
    return n

In [ ]:
n_not_built = solar_network(build_cost=200_000)
n_built = solar_network(build_cost=5_000)

We can see below that the park is either not built (the point at zero) or built with a capacity between 40 MW and 150 MW (`p_nom_min` and `p_nom_max`).
Because `build_cost` is paid only if the park is built, the allowed line jumps from zero to the capital cost at 40 MW plus `build_cost`, and the gap to the dashed line stays at one `build_cost`.
The black rings show the optimum: at zero for the fee of 200,000 € and on the allowed line for the fee of 5,000 €.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
plot_build_mode(axes[0], n_not_built, "solar", "build_cost = 200,000 €: not built")
plot_build_mode(axes[1], n_built, "solar", "build_cost = 5,000 €: built")
fig.tight_layout()

When built, the solar park is then allowed to operate, pushing out peaker plant usage in the middle of the day.
This reduces overall costs as the operational expenditure is larger than the capital expenditure and the grid connection fee.
This is only possible with a sufficiently low build cost.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), width_ratios=[2, 2, 1.2])
plot_dispatch(axes[0], n_not_built, "build_cost = 200,000 €: not built")
plot_dispatch(axes[1], n_built, "build_cost = 5,000 €: built")
plot_costs(
    axes[2], {"not built": cost_summary(n_not_built), "built": cost_summary(n_built)}
)
fig.tight_layout()

### CAUTION: A convex curve caps the capacity

Any piecewise curve caps `p_nom` at its last breakpoint, even if `p_nom_max` is higher.
Here we show the case where we end the piecewise curve at 120 MW and raise `p_nom_max` to 400 MW, which is much higher than the final breakpoint.

In [ ]:
capped_curve = {0: 0, 50: 100, 100: 150, 120: 200}
n_capped = solar_network(build_cost=5_000, capital_cost=capped_curve, p_nom_max=400)

The figure below shows that the park is built at 120 MW, the last breakpoint of the curve, rather than at `p_nom_max`.
The curve defines a cost only up to 120 MW, so PyPSA cannot build beyond it, even though `p_nom_max` allows up to 400 MW.
It is good practice to always extend your piecewise curve definition up to and including `p_nom_max`.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_build_mode(ax, n_capped, "solar", "Convex curve that ends at 120 MW")
ax.axvspan(120, 400, color="#eb6834", alpha=0.15, label="beyond the last breakpoint")
ax.axvline(400, color="0.3", linestyle="--", linewidth=1)
ax.annotate(
    "p_nom_max", (400, 25), xytext=(-6, 0), textcoords="offset points", ha="right"
)
ax.set(xlim=(0, 420), ylim=(0, 30))
ax.legend(loc="lower right", fontsize="small")